<a href="https://colab.research.google.com/github/jhonyjm4/SPRINT2_SERS/blob/main/Sprint2_GoodWe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import time
import random
import pandas as pd
from datetime import datetime

print("=== INICIALIZANDO ECOSSISTEMA CHARGEGRID INTELLIGENCE ===")
print("Carregando firmware: Inversor Híbrido GoodWe (Linha ET)")
print("Conectando ao Smart Meter e Plataforma SEMS...")
print("Aguardando leitura de sensores de campo...\n")
time.sleep(1.5)

# ------------------------------------------------------------------------------
# PARÂMETROS FIXOS DE INFRAESTRUTURA
# ------------------------------------------------------------------------------
LIMITE_CONCESSIONARIA = 100.0  # Limite máximo contratado da rede local (kW)
POTENCIA_MAX_CARREGADOR = 22.0  # Potência nominal de um carregador rápido HCA (kW)

# ------------------------------------------------------------------------------
# ALGORITMO DE CONTROLE DINÂMICO DE CARGA
# ------------------------------------------------------------------------------
def processar_gerenciamento_carga(solar, comercial, veiculos):
    """
    Analisa os dados aleatórios recebidos do ambiente e orquestra a energia.
    """
    # Potência total disponível no Hub (Rede + Geração Solar)
    potencia_total_hub = LIMITE_CONCESSIONARIA + solar

    # Margem que sobra para os carregadores após o prédio comercial consumir
    potencia_disponivel_veiculos = potencia_total_hub - comercial

    # Se não houver veículos, o sistema fica em stand-by
    if veiculos == 0:
        return "Stand-by (Sem veículos)", 0.0, potencia_disponivel_veiculos

    # Potência total necessária se todos carregassem na velocidade máxima
    demanda_solicitada_total = veiculos * POTENCIA_MAX_CARREGADOR

    # Tomada de decisão inteligente baseada nos limites de segurança
    if demanda_solicitada_total <= potencia_disponivel_veiculos:
        status = "Operação Normal - Carga Máxima"
        potencia_alocada_por_veiculo = POTENCIA_MAX_CARREGADOR
    else:
        status = "CONTRAPARTIDA ATIVA - Balanço de Carga"
        # Garante que a divisão não resulte em valores negativos por segurança
        potencia_alocada_por_veiculo = max(0.0, potencia_disponivel_veiculos / veiculos)

    return status, round(potencia_alocada_por_veiculo, 2), round(potencia_disponivel_veiculos, 2)

# ------------------------------------------------------------------------------
# LOOP DE SIMULAÇÃO EM TEMPO REAL (GERAÇÃO DE DADOS ALEATÓRIOS MOCK)
# ------------------------------------------------------------------------------
# Lista para armazenar o histórico de leituras
historico_leituras = []

print("Mapeando ciclos operacionais do Eletroposto Comercial (Pressione Interromper para parar)...")
print("-" * 90)

# Simulando 5 ciclos aleatórios (você pode aumentar ou deixar rodando no vídeo)
for ciclo in range(1, 6):
    hora_atual = datetime.now().strftime("%H:%M:%S")

    # Geração de dados completamente aleatórios simulando a imprevisibilidade do dia a dia
    geracao_solar_atual = round(random.uniform(0.0, 50.0), 2)   # Variação de nuvens/sol (0 a 50 kW)
    consumo_predio_atual = round(random.uniform(30.0, 95.0), 2) # Variação de uso do comércio (30 a 95 kW)
    veiculos_conectados = random.randint(0, 5)                  # De 0 a 5 carros carregando simultaneamente

    # Processamento das informações pelo algoritmo
    status_sistema, pot_por_carro, margem_hub = processar_gerenciamento_carga(
        geracao_solar_atual, consumo_predio_atual, veiculos_conectados
    )

    # Armazenando para gerar o relatório consolidado
    historico_leituras.append({
        "Ciclo": ciclo,
        "Horário": hora_atual,
        "Geração Solar (kW)": geracao_solar_atual,
        "Consumo Predial (kW)": consumo_predio_atual,
        "VEs Ativos": veiculos_conectados,
        "Status Operacional": status_sistema,
        "Entrega/Carro (kW)": pot_por_carro
    })

    # Exibição rápida no console simulando telemetria IoT
    print(f"[{hora_atual}] Ciclo {ciclo} -> Solar: {geracao_solar_atual}kW | Predial: {consumo_predio_atual}kW | Carros: {veiculos_conectados}")
    print(f"            Ação CGI: {status_sistema} -> Liberado: {pot_por_carro} kW por veículo.")
    print("-" * 90)

    time.sleep(2) # Intervalo de 2 segundos entre as leituras de sensores

# ------------------------------------------------------------------------------
# EXIBIÇÃO DO RELATÓRIO FINAL EM DATAFRAME (SERS / SEMS GOODWE)
# ------------------------------------------------------------------------------
print("\n=== CONSOLIDADO DE TELEMETRIA GERADO VIA PLATAFORMA SEMS ===")
df_sems = pd.DataFrame(historico_leituras)
display(df_sems)

# ------------------------------------------------------------------------------
# SIMULAÇÃO DA REAÇÃO DA IA COM BASE NOS RESULTADOS ALEATÓRIOS
# ------------------------------------------------------------------------------
print("\n" + "="*80)
print("INTERAÇÃO ASSISTENTE TÉCNICO CHARGEOPS (CHAIN OF THOUGHTS)")
print("="*80)

# Pega o último cenário gerado aleatoriamente para a IA responder de forma contextualizada
ultimo_cenario = historico_leituras[-1]

print(f"[Contexto Atual do Eletroposto]: {ultimo_cenario['Status Operacional']} com {ultimo_cenario['VEs Ativos']} veículo(s).")
print("Motorista no local pergunta: 'Por que a velocidade de carregamento mudou?'")

print("\n[ChargeOps Cogitação Interna - Árvore de Decisão]:")
print(f"  -> Analisando telemetria do ciclo recente...")
print(f"  -> Consumo Predial: {ultimo_cenario['Consumo Predial (kW)']} kW | Geração Solar: {ultimo_cenario['Geração Solar (kW)']} kW")

if "CONTRAPARTIDA" in ultimo_cenario["Status Operacional"]:
    print("  -> Identificado: Corte ativo de corrente para proteção da infraestrutura local.")
    resposta_ia = (
        f"Olá! Sou o assistente ChargeOps. Detectei que o sistema de Gerenciamento Dinâmico de Carga está atuando. "
        f"Neste exato momento, o consumo comercial local atingiu {ultimo_cenario['Consumo Predial (kW)']} kW. "
        f"Para evitar uma queda do disjuntor geral, nosso Inversor Híbrido GoodWe estabilizou a entrega em "
        f"{ultimo_cenario['Entrega/Carro (kW)']} kW por veículo. A recarga total está garantida e voltará à velocidade máxima "
        f"assim que a demanda comercial diminuir!"
    )
elif ultimo_cenario["VEs Ativos"] == 0:
    print("  -> Identificado: Sistema operando em stand-by estável.")
    resposta_ia = "Olá! Sou o ChargeOps. O hub está operando em modo de espera com máxima eficiência energética. Pronto para receber novas recargas!"
else:
    print("  -> Identificado: Operação estável com folga energética.")
    resposta_ia = (
        f"Olá! Sou o ChargeOps. Tudo operando perfeitamente por aqui! O estabelecimento está com consumo controlado e a "
        f"geração solar está auxiliando o sistema. Seu veículo está recebendo a potência máxima permitida de "
        f"{ultimo_cenario['Entrega/Carro (kW)']} kW."
    )

print("\n[Resposta do Chatbot ChargeOps]:")
print(resposta_ia)

=== INICIALIZANDO ECOSSISTEMA CHARGEGRID INTELLIGENCE ===
Carregando firmware: Inversor Híbrido GoodWe (Linha ET)
Conectando ao Smart Meter e Plataforma SEMS...
Aguardando leitura de sensores de campo...

Mapeando ciclos operacionais do Eletroposto Comercial (Pressione Interromper para parar)...
------------------------------------------------------------------------------------------
[22:44:27] Ciclo 1 -> Solar: 1.28kW | Predial: 76.51kW | Carros: 1
            Ação CGI: Operação Normal - Carga Máxima -> Liberado: 22.0 kW por veículo.
------------------------------------------------------------------------------------------
[22:44:29] Ciclo 2 -> Solar: 6.73kW | Predial: 45.52kW | Carros: 0
            Ação CGI: Stand-by (Sem veículos) -> Liberado: 0.0 kW por veículo.
------------------------------------------------------------------------------------------
[22:44:31] Ciclo 3 -> Solar: 6.89kW | Predial: 87.09kW | Carros: 1
            Ação CGI: CONTRAPARTIDA ATIVA - Balanço de Carga ->

,Ciclo,Horário,Geração Solar (kW),Consumo Predial (kW),VEs Ativos,Status Operacional,Entrega/Carro (kW)
0,1,22:44:27,1.28,76.51,1,Operação Normal - Carga Máxima,22.00
1,2,22:44:29,6.73,45.52,0,Stand-by (Sem veículos),0.00
2,3,22:44:31,6.89,87.09,1,CONTRAPARTIDA ATIVA - Balanço de Carga,19.80
3,4,22:44:33,41.30,35.61,4,Operação Normal - Carga Máxima,22.00
4,5,22:44:35,49.44,77.71,5,CONTRAPARTIDA ATIVA - Balanço de Carga,14.35



INTERAÇÃO ASSISTENTE TÉCNICO CHARGEOPS (CHAIN OF THOUGHTS)
[Contexto Atual do Eletroposto]: CONTRAPARTIDA ATIVA - Balanço de Carga com 5 veículo(s).
Motorista no local pergunta: 'Por que a velocidade de carregamento mudou?'

[ChargeOps Cogitação Interna - Árvore de Decisão]:
  -> Analisando telemetria do ciclo recente...
  -> Consumo Predial: 77.71 kW | Geração Solar: 49.44 kW
  -> Identificado: Corte ativo de corrente para proteção da infraestrutura local.

[Resposta do Chatbot ChargeOps]:
Olá! Sou o assistente ChargeOps. Detectei que o sistema de Gerenciamento Dinâmico de Carga está atuando. Neste exato momento, o consumo comercial local atingiu 77.71 kW. Para evitar uma queda do disjuntor geral, nosso Inversor Híbrido GoodWe estabilizou a entrega em 14.35 kW por veículo. A recarga total está garantida e voltará à velocidade máxima assim que a demanda comercial diminuir!
